# 22 — Data Transformation for Classification (Objective 2)

**Objective.** Turn the preprocessed warehouse table into a table a classifier can learn from, without letting the test
set influence anything. Five things happen here, in this order:

| § | Step |
|---|---|
| 1 | Build the modelling table: key, target, candidate features |
| 2 | **Assign every warehouse to training or test — once** |
| 3 | Re-check the target banding on the training split |
| 4 | Decide how establishment year is represented |
| 5 | Encode the categorical features |
| 6 | Specify scaling and any shape transformation |
| 7–8 | Save and check |

**Binary target.** The classifier predicts **Not High Risk (0–3 breakdowns) vs High Risk (4–6 breakdowns)**. The
cutoff at 3|4 is the single class boundary confirmed in the EDA and re-checked on training warehouses in §3.

**Why the split comes first.** The train/test split is placed here rather than in the modelling step, so that from §2
onward every figure used to make a decision is computed on training warehouses only. The 5,000 test warehouses are carried
along untouched, and are first used to measure performance in the results notebook.

**What is and is not learned from the data.** Encoding in this notebook follows fixed, written-out rules — level lists and
grade orders from the data dictionary — so it learns nothing from any row and can be applied to training and test alike.
Scalers *do* learn from the data (a mean and a spread), so this notebook only **specifies** scaling. The modelling step
fits the scaler on training warehouses, inside each cross-validation fold.

**Input:** `data/preprocessed/warehouse_preprocessed.csv`
**Output:** `data/processed/classification_base.csv` · `feature_engine/feature_roles.csv` · `feature_engine/feature_spec.md`

## 0. Setup

In [ ]:
import sys, pathlib

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "src" / "common.py").exists())
sys.path.insert(0, str(ROOT))
from src.common import *

set_style()
paths = obj_paths(2)

CLASSES = ["Not High Risk", "High Risk"]

df = load_preprocessed()
print(f"loaded {df.shape[0]:,} rows x {df.shape[1]} columns from {PREPROCESSED_FILE.name}")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score

---
## 1. The modelling table

The column roles are those established in the EDA, and the target banding is **Not High Risk 0–3 · High Risk 4–6**
breakdowns in three months. All 23 candidate features are carried forward; a characteristics-only list is recorded in §7
for a characteristics-only model comparison.

In [ ]:
key = "Ware_house_ID"
target_source = "wh_breakdown_l3m"
period_measures = ["product_wg_ton", "storage_issue_reported_l3m", "num_refill_req_l3m",
                   "govt_check_l3m", "transport_issue_l1y"]
characteristics_numeric = ["wh_est_year", "workers_num", "dist_from_hub", "Competitor_in_mkt",
                           "retail_shop_num", "distributor_num"]
characteristics_binary = ["electric_supply", "temp_reg_mach", "flood_proof", "flood_impacted", "is_unrated_warehouse"]
characteristics_categorical = ["approved_wh_govt_certificate", "Location_type", "WH_capacity_size",
                               "WH_regional_zone", "zone", "wh_owner_type"]
recording_flags = ["wh_est_year_missing"]

candidates = (period_measures + characteristics_numeric + characteristics_binary
              + characteristics_categorical + recording_flags)
assert sorted([key, target_source] + candidates) == sorted(df.columns)
assert len(candidates) == 23

TARGET_BINS = [-0.5, 3.5, 6.5]
df["breakdown_risk"] = pd.cut(df[target_source], bins=TARGET_BINS, labels=CLASSES, ordered=True)
assert df["breakdown_risk"].notna().all()

counts = df["breakdown_risk"].value_counts().reindex(CLASSES)
print(f"candidate features: {len(candidates)}\n")
pd.DataFrame({"warehouses": counts, "pct": (100 * counts / len(df)).round(2)})

> **Interpretation.**
>
> - The table holds all 25,000 warehouses with the 23 candidate features established in the EDA. The binary target groups
>   0–3 breakdowns as **Not High Risk (13,026; 52.10%)** and 4–6 breakdowns as **High Risk (11,974; 47.90%)** —
>   nearly balanced at a ratio of approximately 1.09:1.

---
## 2. The train/test split

**Settings.**

- Training share: 80%.
- Test share: 20%.
- Split type: stratified by risk class.
- Random seed: `random_state = 42` (`RANDOM_STATE` in `common.py`).

**Why.**

- A 5,000-row test set holds 2,395 High Risk and 2,605 Not High Risk warehouses — both large enough for stable
  performance estimates.
- The classes are nearly balanced (ratio ~1.09:1), so stratification here mainly ensures reproducibility.
- The 20,000-row training set remains large enough for model fitting.
- A fixed random seed makes the split reproducible.

**Check below.**

- Compare class shares in train, test and the full network.
- Count the thin groups flagged in the EDA:
  - unrated warehouses;
  - `zone = East`.

> **Decision — train/test split.**
>
> - **Choice:** 80/20 stratified split by risk class, fixed random seed (`RANDOM_STATE = 42`).
> - **Numbers:** 20,000 training rows and 5,000 test rows. Not High Risk: 10,421 train / 2,605 test; High Risk: 9,579 / 2,395.
> - **Rejected:** a larger test fraction (25%) — reduces training set size unnecessarily given the near-balanced classes; an unstratified split — risks slightly misrepresenting one class in either part by chance.

In [ ]:
train_idx, test_idx = train_test_split(df.index, test_size=0.20, stratify=df["breakdown_risk"],
                                       random_state=RANDOM_STATE)
df["split"] = "train"
df.loc[test_idx, "split"] = "test"

sizes = pd.crosstab(df["breakdown_risk"], df["split"])[["train", "test"]]
sizes.index = sizes.index.astype(str)
sizes["total"] = sizes.sum(axis=1)
sizes.loc["total"] = sizes.sum()
shares = 100 * pd.crosstab(df["breakdown_risk"], df["split"], normalize="columns")[["train", "test"]]
shares["whole network"] = 100 * df["breakdown_risk"].value_counts(normalize=True)
shares["largest gap (pp)"] = shares[["train", "test"]].sub(shares["whole network"], axis=0).abs().max(axis=1)

print("warehouses per class and split:")
print(sizes.to_string())
print("\nclass shares (%):")
print(shares.round(3).to_string())

thin = pd.DataFrame({
    "unrated warehouses": df.groupby("split")["is_unrated_warehouse"].sum(),
    "zone = East": df[df["zone"] == "East"].groupby("split").size(),
    "zone = East, Not High Risk": df[(df["zone"] == "East") & (df["breakdown_risk"] == "Not High Risk")].groupby("split").size(),
    "Urban, Not High Risk": df[(df["Location_type"] == "Urban") & (df["breakdown_risk"] == "Not High Risk")].groupby("split").size(),
}).T[["train", "test"]]
print("\nthin groups in each split:")
thin

> **Interpretation.**
>
> - The split is **20,000 training / 5,000 test warehouses**, and stratification did its job. Each class
>   holds the same share in both parts as in the whole network, to within a few hundredths of a percentage point:
>
> | Class | Train | Test |
> |---|---|---|
> | Not High Risk | 10,421 | 2,605 |
> | High Risk | 9,579 | 2,395 |
>
> - The classes are **nearly balanced (ratio ~1.09:1)**, so macro F1 and accuracy give similar signals. No resampling
>   is expected.
>
> - The thin groups divide as expected: 740 unrated warehouses in training and 168 in test, and 345 East-zone warehouses in
>   training and 84 in test. Urban Not High Risk warehouses number approximately 94 in training and 24 in test.
>
> - From here on, every figure used to make a decision comes from the 20,000 training warehouses.

---
## 3. Safeguard check — does the target banding boundary hold on training warehouses alone?

The target banding's boundary was chosen after viewing all 25,000 warehouses in the EDA. The safeguard set out at that
point repeats that measurement **on the training split only**, and fixes the rule in advance:

> The banding is revisited if a step **inside** a class (0 vs 1, 1 vs 2, 2 vs 3, 4 vs 5, or 5 vs 6) is larger than the
> **boundary** step (3 vs 4) on any of the three features that move with breakdowns.

η² is the share of a feature's variance explained by which of the two neighbouring counts a warehouse has; 0 means the two
counts do not differ. Establishment year uses recorded years only, as in the EDA.

This is the last use of `wh_breakdown_l3m`. Once the check is done it is removed from the table, so the count the target was
built from cannot reach the features.

In [ ]:
train = df[df["split"] == "train"]
followed = ["product_wg_ton", "storage_issue_reported_l3m", "wh_est_year"]

def eta_sq(data, feature, k):
    d = data[data["wh_est_year_missing"] == 0] if feature == "wh_est_year" else data
    pair = d[d[target_source].isin([k, k + 1])]
    vals = pair[feature]; groups = pair[target_source]
    gm = vals.mean()
    between = sum(len(v) * (v.mean() - gm) ** 2 for _, v in vals.groupby(groups, observed=True))
    return between / ((vals - gm) ** 2).sum()

not_high_risk_steps = [0, 1, 2]   # 0|1, 1|2, 2|3 — within Not High Risk
high_risk_steps     = [4, 5]      # 4|5, 5|6 — within High Risk
boundary_step       = 3           # 3|4 — the single class boundary

rows = []
for k in not_high_risk_steps + [boundary_step] + high_risk_steps:
    role = ("within Not High Risk" if k in not_high_risk_steps else
            "BOUNDARY Not High Risk | High Risk" if k == boundary_step else
            "within High Risk")
    row = {"step": f"{k} vs {k + 1}", "role in banding": role}
    for c in followed:
        row[c] = eta_sq(train, c, k)
    rows.append(row)

steps = pd.DataFrame(rows).set_index("step")
print("eta squared between neighbouring breakdown counts - TRAINING warehouses only:")
print(steps.round(4).to_string())

within_steps = not_high_risk_steps + high_risk_steps
rule_df = pd.DataFrame({
    "largest within-class step": steps.loc[[f"{k} vs {k+1}" for k in within_steps], followed].max(),
    "boundary step (3|4)": steps.loc["3 vs 4", followed],
})
rule_df["boundary holds"] = rule_df.iloc[:, 0] < rule_df.iloc[:, 1]
rule_df["boundary / within-class max"] = rule_df.iloc[:, 1] / rule_df.iloc[:, 0]
print("\nSafeguard rule — boundary step (3|4) exceeds all within-class steps?")
print(rule_df.round(4).to_string())
print(f"\nBoundary 3|4 is the largest step on the training split for all three features: {bool(rule_df['boundary holds'].all())}")

df = df.drop(columns=target_source)
assert target_source not in df.columns
print(f"'{target_source}' removed from the table")

> **Interpretation.**
>
> - **The safeguard, read as written, does not pass.** The rule fixed in advance asked whether the 3|4 boundary step is
>   the largest of all six adjacent steps. It is not — the cell above prints `boundary holds: False` for all three
>   features, because two steps inside Not High Risk are much larger.
>
> | Step | Role in banding | η² shipment / storage issues / year |
> |---|---|---|
> | 0 vs 1 | within Not High Risk | 0.2051 / 0.5069 / 0.4027 |
> | 1 vs 2 | within Not High Risk | 0.1597 / 0.1698 / 0.2176 |
> | 2 vs 3 | within Not High Risk | 0.0003 / 0.0003 / 0.0002 |
> | 3 vs 4 | **BOUNDARY** | **0.0266 / 0.0282 / 0.0384** |
> | 4 vs 5 | within High Risk | 0.0008 / 0.0008 / 0.0010 |
> | 5 vs 6 | within High Risk | 0.0001 / 0.0001 / 0.0002 |
>
> - **Why those two steps do not argue for moving the boundary.** The 0|1 and 1|2 steps sit on the ramp out of the 908
>   newly commissioned warehouses, which record zero storage issues, zero breakdowns and about a quarter of the typical
>   shipment volume (NB 00 §10). Cutting there is exactly the naive "any breakdown" target, which NB 21 §3.2 measures as
>   26.53 : 1 imbalanced and — decisively — as keeping **0.0% of the signal among rated warehouses**. That step separates
>   not-yet-operating warehouses from operating ones. It is a commissioning status, not a risk gradient.
>
> - **Among the steps that actually split operating warehouses, 3|4 is the strongest by a wide margin:** 0.0266 / 0.0282
>   / 0.0384, against a largest rival of 0.0008 / 0.0008 / 0.0010 at 4|5 — a factor of roughly 33 to 38.
>
> - The training-only boundary figures sit close to the EDA's figures on all 25,000 warehouses (3|4 there: 0.0235 /
>   0.0249 / 0.0351), so the boundary is a property of the warehouses, not of the test rows the EDA could see.
>
> - **The banding is kept**, on the narrower claim above rather than on the rule as originally worded.
>   `wh_breakdown_l3m` has been removed from the table.

---
## 4. How establishment year is represented

**Starting point.**

- The preprocessing step filled every missing `wh_est_year` with 2009.
- The preprocessing step added `wh_est_year_missing`.
- The EDA found that the filled year still carries risk signal.
- The EDA also found that the missing flag carries almost no signal by itself.

**Tension to resolve.**

- Filling costs the year about half its signal: η² 0.1002 on recorded years, 0.0525 after filling (NB 21 §4).
- The flag alone carries almost no information about risk; the measurement is repeated below.
- A flag that says little alone can still matter together with the year.
- Without the flag, a model cannot distinguish a true 2009 warehouse from an unknown-year warehouse filled as 2009.

**Three training-only measurements.**

1. Count the filled rows and the genuine 2009 rows.
2. Compare risk-class shares for:
   - missing year;
   - genuine 2009;
   - any other recorded year.
3. Compare mutual information for:
   - the flag alone;
   - the filled year alone;
   - the year with missing values kept apart as their own category.

In [ ]:
train = df[df["split"] == "train"]
missing = train["wh_est_year_missing"] == 1

print(f"training warehouses with the year missing : {int(missing.sum()):,} ({100 * missing.mean():.2f}%)")
print(f"distinct year values among them           : {sorted(train.loc[missing, 'wh_est_year'].unique())}")
print(f"recorded warehouses genuinely from 2009   : {int(((~missing) & (train['wh_est_year'] == 2009)).sum()):,}\n")

group = pd.Series(np.select([missing, train["wh_est_year"] == 2009], ["year missing (filled as 2009)", "recorded as 2009"],
                            default="recorded, any other year"), index=train.index, name="group")
by_group = 100 * pd.crosstab(group, train["breakdown_risk"], normalize="index")[CLASSES]
by_group.insert(0, "warehouses", group.value_counts())
print("risk-class shares by group (%):")
print(by_group.round(2).to_string())

y = train["breakdown_risk"].astype(str)
class_uncertainty = mutual_info_score(y, y)
versions = {
    "flag alone": train["wh_est_year_missing"],
    "filled year alone (as NB 01 left it)": train["wh_est_year"],
    "year with missing kept apart (filled year + flag)": np.where(missing, "missing", train["wh_est_year"].astype(str)),
}
mi = pd.DataFrame({
    "categories": [pd.Series(v).nunique() for v in versions.values()],
    "mutual_information_bits": [mutual_info_score(y, v) / np.log(2) for v in versions.values()],
    "pct_of_class_uncertainty": [100 * mutual_info_score(y, v) / class_uncertainty for v in versions.values()],
}, index=list(versions))
print(f"\nclass uncertainty (entropy): {class_uncertainty / np.log(2):.4f} bits")
mi.round(4)

> **Interpretation.**
>
> - **What the filled rows are.** 9,516 training warehouses (47.58%) have no recorded year, and every one of them carries
>   exactly **2009**. Only **391** training warehouses were genuinely established in 2009. So without the flag, the value 2009
>   covers 9,907 warehouses, of which the genuine ones are outnumbered by roughly 24 to 1.
>
> - **Whether the distinction matters for risk.** It does.
>
> | Group | Warehouses | Not High Risk | High Risk |
> |---|---|---|---|
> | recorded as 2009 | 391 | **40.41%** | **59.59%** |
> | recorded, any other year | 10,093 | 52.58% | 47.42% |
> | year missing, filled as 2009 | 9,516 | 52.08% | 47.92% |
>
> - Warehouses genuinely established in 2009 are markedly higher-risk: **59.59% High Risk**, against 47.92% for the
>   unknown-year group. The unknown-year warehouses look like the network as a whole. Filled with the same value, the
>   unknowns drown out the real 2009 group's profile.
>
> - **Mutual information confirms it — and shows where the flag's value lies.** Class uncertainty is 0.9987 bits.
>
> - **The flag alone:** 0.0000 bits — no measurable information about risk, matching the EDA (V 0.0045).
> - **The filled year alone:** 0.0760 bits, **7.61%** of the class's uncertainty.
> - **The year with missing values kept apart:** 0.0767 bits, **7.68%** — what the filled year and the flag convey together.
>
> - The gain is small (0.07 percentage points), but it is real, and it exists only in combination. The flag says nothing
>   about risk by itself; it tells a model when *not* to trust the year.

> **Decision — establishment year representation: keep the filled year together with `wh_est_year_missing`.**
>
> **Why kept.**
>
> - The flag separates 9,516 unknown years from 391 genuine 2009 warehouses.
> - Their risk profiles differ sharply: **59.59% High Risk** for the genuine 2009 group against **47.92%** for the
>   unknown-year group.
> - The year carries 7.68% of the class's uncertainty with the flag, against 7.61% without it.
> - For a linear model, the flag gives unknown-year warehouses their own offset instead of treating them as true 2009 warehouses.
> - For a tree model, the flag lets splits on year set unknown-year warehouses aside.
> - The cost is one 0/1 column already present in the data.
>
> **How it is judged.**
>
> - The flag's value exists only alongside `wh_est_year`.
> - On its own, the flag carries no measurable mutual information (0.0000 bits).
> - Because it is a companion to the filled year, it is not part of the feature engineering screen for negligible features.
>
> **Rejected.**
>
> - Drop the flag: merges the unknown-year group with true 2009 warehouses.
> - Leave missing years as gaps: reverses the preprocessing decision to fill them, and the logistic regression and SVM
>   estimators used in the screening step cannot accept missing values.

In [ ]:
# Establishment-year representation, as the setting used below
recording_flags_kept = recording_flags
D4_SPEC = (f"- `wh_est_year` as filled by NB 01, kept together with `wh_est_year_missing`.\n"
           f"- Mutual information with the risk class, training split: flag alone "
           f"{mi.loc['flag alone', 'pct_of_class_uncertainty']:.2f}% of class uncertainty; filled year alone "
           f"{mi.loc['filled year alone (as NB 01 left it)', 'pct_of_class_uncertainty']:.2f}%; year with missing kept apart "
           f"{mi.loc['year with missing kept apart (filled year + flag)', 'pct_of_class_uncertainty']:.2f}%.\n"
           f"- Judged together with the year, so not subject to NB 23's one-feature screen.")
print("recording flags kept:", recording_flags_kept)

---
## 5. Encoding the categorical features

**Encoding rule.**

| Encoding | Use when | Why |
|---|---|---|
| label / 0-1 | category has two levels | one column fully represents the feature |
| ordinal | levels have a natural order | one column preserves the order |
| one-hot | levels have no natural order | avoids inventing a false order |

**Proposed encodings.**

- `approved_wh_govt_certificate` → ordinal grade:
  - Unrated = 0;
  - C = 1;
  - B = 2;
  - B+ = 3;
  - A = 4;
  - A+ = 5.
- Read `Unrated = 0` together with `is_unrated_warehouse`.
- `WH_capacity_size` → ordinal size:
  - Small = 1;
  - Mid = 2;
  - Large = 3.
- `Location_type` and `wh_owner_type` → 0/1 columns.
- `zone` and `WH_regional_zone` → one-hot columns.
- Leave out one reference level for each nominal feature.
- Use the most common training level as the reference, so comparisons are made against a stable base.

**Checks below.**

- Confirm that risk moves in a consistent direction across certificate grades among rated training warehouses.
- Identify the most common training level for each nominal feature.

In [ ]:
train = df[df["split"] == "train"]
grade_order = ["C", "B", "B+", "A", "A+"]
rated = train[train["is_unrated_warehouse"] == 0]

by_grade = 100 * pd.crosstab(rated["approved_wh_govt_certificate"], rated["breakdown_risk"], normalize="index")
by_grade = by_grade.reindex(grade_order)[CLASSES]
by_grade.insert(0, "warehouses", rated["approved_wh_govt_certificate"].value_counts().reindex(grade_order))
grade_rank = rated["approved_wh_govt_certificate"].map({g: i for i, g in enumerate(grade_order, start=1)})
print("risk-class shares by certificate grade — rated TRAINING warehouses (%):")
print(by_grade.round(2).to_string())
print(f"\nSpearman correlation, grade order vs risk-class order: "
      f"{grade_rank.corr(rated['breakdown_risk'].cat.codes, method='spearman'):.4f}")

print("\nlevel counts in the training split (most common level first):")
for c in ["Location_type", "wh_owner_type", "zone", "WH_regional_zone"]:
    vc = train[c].value_counts()
    print(f"  {c:<18} " + " · ".join(f"{level} {n:,}" for level, n in vc.items()))

> **Interpretation.**
>
> - **Risk moves with certificate grade** among rated training warehouses — the Not High Risk share falls and the High Risk
>   share rises as the grade improves:
>
> | Grade | Warehouses | Not High Risk | High Risk |
> |---|---|---|---|
> | C | 4,413 | ~57.44% | ~42.56% |
> | B | 3,840 | ~49.77% | ~50.23% |
> | B+ | 3,939 | ~50.14% | ~49.86% |
> | A | 3,710 | ~48.12% | ~51.89% |
> | A+ | 3,358 | ~43.93% | ~56.08% |
>
> - The Not High Risk share falls from ~57.44% at grade C to ~43.93% at grade A+; High Risk rises from ~42.56% to
>   ~56.08%. The Spearman correlation between grade order and risk order is weak but positive.
>
> - An ordinal code assumes a direction, not a strong effect, and the direction is there. A counter-intuitive detail worth
>   recording: **better-graded warehouses carry more breakdown risk here.** That is an association in a single snapshot,
>   not a claim that a better grade causes breakdowns. Why it occurs is not examined in this objective.
>
> - **Most common levels in the training split**, which become the references: Rural (18,369 of 20,000), Company Owned
>   (10,837), North (8,261) and Zone 6 (6,700). East is the thinnest zone, at 345.

> **Decision — encoding.**
>
> | Feature | Encoding | Result |
> |---|---|---|
> | `approved_wh_govt_certificate` | **ordinal** — Unrated 0 · C 1 · B 2 · B+ 3 · A 4 · A+ 5, read with `is_unrated_warehouse` | `certificate_grade` |
> | `WH_capacity_size` | **ordinal** — Small 1 · Mid 2 · Large 3 | `capacity_size` |
> | `Location_type` | 0/1, reference Rural | `Location_type_Urban` |
> | `wh_owner_type` | 0/1, reference Company Owned | `wh_owner_type_Rented` |
> | `zone` | **one-hot**, reference North | `zone_East`, `zone_South`, `zone_West` |
> | `WH_regional_zone` | **one-hot**, reference Zone 6 | `WH_regional_zone_Zone_1` … `_Zone_5` |
>
> **Why these encodings.**
>
> - Certificate grades have an official order, and the Low-risk share falls at every grade step.
> - One ordinal certificate column keeps that order.
> - One-hot certificate encoding would use five columns for a feature with a negligible rated-warehouse effect (V 0.0707).
> - The most common training level is used as the reference for nominal features.
> - A well-populated reference makes dummy-column estimates steadier.
> - Zones and regional zones have no order, so numbered codes would invent one.
> - East is kept as its own column because there is no evidence for a particular partner zone to merge it with.
>
> **Implementation.**
>
> - The encoding rules are fixed in writing.
> - The same rules are applied to training and test warehouses.
> - The encoding does not learn from either split.
> - The 23 candidate inputs become **29 feature columns**.
>
> **Rejected.**
>
> - One-hot certificate: five columns and the grade order is discarded.
> - Numbered codes for zone and regional zone: creates a false order.
> - Merging East into another zone: no evidence for any particular partner.

In [ ]:
GRADE = {"Unrated": 0, "C": 1, "B": 2, "B+": 3, "A": 4, "A+": 5}
CAPACITY = {"Small": 1, "Mid": 2, "Large": 3}
LEVELS = {                                   # every level written out, so nothing is learned from the rows
    "Location_type": ["Rural", "Urban"],
    "wh_owner_type": ["Company Owned", "Rented"],
    "zone": ["East", "North", "South", "West"],
    "WH_regional_zone": ["Zone 1", "Zone 2", "Zone 3", "Zone 4", "Zone 5", "Zone 6"],
}
REFERENCE = {c: train[c].value_counts().idxmax() for c in LEVELS}      # most common level, training split only

assert set(df["approved_wh_govt_certificate"]) == set(GRADE)
assert set(df["WH_capacity_size"]) == set(CAPACITY)
for c, levels in LEVELS.items():
    assert set(df[c]) == set(levels), c

base = df[[key, "split", "breakdown_risk"] + period_measures + characteristics_numeric
          + characteristics_binary + recording_flags_kept].copy()
base["certificate_grade"] = df["approved_wh_govt_certificate"].map(GRADE)
base["capacity_size"] = df["WH_capacity_size"].map(CAPACITY)

dummy_source = {}
for c, levels in LEVELS.items():
    for level in levels:
        if level != REFERENCE[c]:
            name = f"{c}_{level}".replace(" ", "_")
            base[name] = (df[c] == level).astype(int)
            dummy_source[name] = c

features = [c for c in base.columns if c not in (key, "split", "breakdown_risk")]
print("reference levels (left out):", REFERENCE)
print(f"\nencoded feature columns: {len(features)}")
print("  new ordinal columns :", ["certificate_grade", "capacity_size"])
print("  new 0/1 columns     :", list(dummy_source))

---
## 6. Scaling and shape

**Which models need scaling.** This follows from how each algorithm works, not from the data:

| Model | Affected by units? | Why |
|---|---|---|
| logistic regression | **yes** | its regularisation penalises every coefficient alike, so a feature measured in large units is penalised less for the same effect, and the solver converges poorly |
| support vector machine | **yes** | built on distances between warehouses, which large-unit features dominate |
| Gaussian Naive Bayes | no | fits a separate mean and spread per feature and class; rescaling a feature changes nothing |
| decision tree, random forest, AdaBoost, XGBoost, CatBoost | no | split one feature at a time at a threshold; rescaling moves the threshold, not the split |

**Which columns, and which scaler.** The table below profiles every encoded column on training warehouses under the two
usual candidates:

- **`StandardScaler`** — subtract the mean, divide by the standard deviation. Every column ends with spread 1.
- **`RobustScaler`** — subtract the median, divide by the interquartile range (IQR). It resists extreme values, but a
  column whose IQR is zero cannot be scaled this way at all.

It also records skew: whether any column is lopsided enough to need a shape transformation such as a log, before
scaling.

In [ ]:
train_base = base[base["split"] == "train"]
kind = {c: ("period measure" if c in period_measures else
            "numeric characteristic" if c in characteristics_numeric else
            "ordinal" if c in ("certificate_grade", "capacity_size") else
            "0/1 flag" if c in characteristics_binary + recording_flags else "one-hot") for c in features}

rows = []
for c in features:
    s = train_base[c].astype(float)
    z = (s - s.mean()) / s.std(ddof=0)
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    robust = (s - s.median()) / iqr if iqr > 0 else None
    rows.append({
        "column": c, "kind": kind[c], "distinct": s.nunique(), "skew": s.skew(),
        "standard: max |value|": z.abs().max(), "standard: % beyond 3": 100 * (z.abs() > 3).mean(),
        "IQR": iqr,
        "robust: resulting spread (sd)": np.nan if robust is None else robust.std(ddof=0),
        "robust: max |value|": np.nan if robust is None else robust.abs().max(),
    })
profile = pd.DataFrame(rows).set_index("column")

print("columns with IQR = 0 (RobustScaler cannot scale them):",
      list(profile.index[profile["IQR"] == 0]), "\n")
profile.round(3)

> **Interpretation.**
>
> - **The 13 numeric and ordinal columns.**
>
> - **`StandardScaler`** gives every one a spread of 1. Its most extreme values are **9.006** for `workers_num`, 7.823 for
>   `Competitor_in_mkt`, 5.716 for `retail_shop_num` and 3.529 for `transport_issue_l1y`. No column has more than
>   **1.405%** of training warehouses beyond 3 standard deviations; the highest is `retail_shop_num`.
> - **`RobustScaler`** leaves the spreads unequal: from 0.496 (`certificate_grade`) to 1.198 (`transport_issue_l1y`) — and
>   **5.47 for `wh_est_year`**. The filled year's interquartile range is just **1**, because nearly half the training
>   warehouses sit at 2009 (§4), so the middle half of its values spans only one year. Divided by that, the year would
>   reach values as far as **14.0** and dominate every distance and penalty — an artefact of NB 01's filling, not a
>   property of the warehouses.
>
> - **The 16 on/off columns.** 10 of them have an interquartile range of **0** and cannot be robust-scaled at all. Standard
>   scaling would turn a rare flag into large values: 7.548 for `zone_East`, 5.102 for `is_unrated_warehouse`, 4.147 for
>   `flood_proof`.
>
> - **Shape.** Among the 13 numeric and ordinal columns the largest skew is **1.610** (`transport_issue_l1y`), followed by
>   1.058 (`workers_num`), 0.963 (`Competitor_in_mkt`) and 0.912 (`retail_shop_num`). Every other column's skew is 0.373 or
>   less in absolute size.


> **Decision — scaling.**
>
> - Use `StandardScaler` on the 13 numeric and ordinal columns.
> - Apply scaling only for logistic regression and SVM.
> - Use PyCaret normalization in the modelling step for model-family screening; the final AdaBoost model uses the encoded table without a manual scaler.
> - Do not let a fold's scaler see that fold's validation rows.
>
> **Why `StandardScaler`.**
>
> - It gives every scaled column the same spread.
> - `RobustScaler` would give the filled year a spread of 5.47 against 0.50–1.20 for the rest.
> - In this case the scaler meant to resist extremes would create one.
>
> **Why 0/1 columns stay unscaled.**
>
> - Their coefficients remain readable as the effect of having the condition.
> - 10 of the 16 cannot be robust-scaled at all.
> - Standard scaling would inflate rare flags to values as large as 7.548.
> - Unscaled, they already share the numeric columns' order of magnitude.
>
> **Why only logistic regression and SVM.**
>
> - Tree models, including the selected AdaBoost, give the same predictions with or without scaling.
>
> **Rejected.**
>
> - `RobustScaler`: distorts the filled year.
> - Scaling the 0/1 columns: inflates rare flags and loses readable coefficients.
> - A project-wide scaler fitted here would let later held-out rows influence the screen, so scaling is left inside PyCaret.

> **Decision — no shape transformation.**
>
> **Why.**
>
> - The largest skew among columns that would be scaled is 1.610.
> - No such column has more than 1.405% of training warehouses beyond 3 standard deviations.
> - The extreme values were judged real and retained in the preprocessing step.
> - Tree models are unaffected by shape transformation.
> - `transport_issue_l1y` is the most skewed scaled column, but it is a 0–5 count.
> - A log transform would treat the step from 4 to 5 issues as smaller than the step from 0 to 1.
> - That would weaken the simple count meaning.
>
> **Rejected.**
>
> - Log transforms of `transport_issue_l1y`, `workers_num`, `Competitor_in_mkt` or `retail_shop_num`.
> - Square-root transforms of the same columns.

In [ ]:
# Scaling and shape settings, as saved in §7
SCALE_COLUMNS = period_measures + characteristics_numeric + ["certificate_grade", "capacity_size"]
to_scale = profile.loc[SCALE_COLUMNS]
others = to_scale.drop(index="wh_est_year")

SCALER_SPEC = (f"`StandardScaler` — every scaled column ends with spread 1. `RobustScaler` rejected: filled `wh_est_year` "
               f"has IQR {profile.loc['wh_est_year', 'IQR']:g}, which would leave it with spread "
               f"{profile.loc['wh_est_year', 'robust: resulting spread (sd)']:.2f} against "
               f"{others['robust: resulting spread (sd)'].min():.2f}–{others['robust: resulting spread (sd)'].max():.2f} "
               f"for the other scaled columns.")
SHAPE_SPEC = (f"- None. Largest skew among scaled columns {to_scale['skew'].max():.3f} (`{to_scale['skew'].idxmax()}`); "
              f"at most {to_scale['standard: % beyond 3'].max():.3f}% of training warehouses beyond 3 standard deviations "
              f"in any of them; extremes retained as real.")

print(f"columns to scale for logistic regression and SVM ({len(SCALE_COLUMNS)}):", SCALE_COLUMNS)
print("scaler:", SCALER_SPEC)
print("shape :", SHAPE_SPEC)

---
## 7. Save

- **`classification_base.csv`** — all 25,000 warehouses: key, split, target and the encoded features, **unscaled**.
  PyCaret screening in NB 24 handles normalization inside the model-family screen.
- **`feature_roles.csv`** — one row per feature: the source column, its role (NB 21 §2), its encoding, whether it is scaled
  for logistic regression and SVM, and whether it belongs to the **characteristics-only** feature set.
  Later notebooks read their column lists from this file instead of retyping them.
- **`feature_spec.md`** — the same record in words, written from this notebook's variables so its numbers match what was
  saved.

In [ ]:
source = {c: c for c in features}
source.update({"certificate_grade": "approved_wh_govt_certificate", "capacity_size": "WH_capacity_size"})
source.update(dummy_source)

role_of = {}
for c in features:
    s = source[c]
    role_of[c] = ("period measure" if s in period_measures else
                  "recording flag" if s in recording_flags else "characteristic")

roles = pd.DataFrame({
    "feature": features,
    "source_column": [source[c] for c in features],
    "role": [role_of[c] for c in features],
    "encoding": [kind[c] if kind[c] in ("ordinal", "one-hot") else
                 "0/1" if kind[c] == "0/1 flag" else "numeric, as recorded" for c in features],
    "scale_for_lr_svm": [c in SCALE_COLUMNS for c in features],
    "characteristics_only_set": [role_of[c] != "period measure" for c in features],
})

save_table(base, paths["processed"] / "classification_base.csv", index=False)
save_table(roles, paths["feature_engine"] / "feature_roles.csv", index=False)

split_sizes = base["split"].value_counts()
spec = f'''# Objective 2 — classification feature specification

Written by `notebooks/22_data_transformation.ipynb`. Updated by NB 23 if features are added or removed.

## Rows and target
- **{len(base):,} warehouses**, all kept.
- **Target `breakdown_risk`**: Not High Risk 0–3 · High Risk 4–6 breakdowns in three months.
  The source count `wh_breakdown_l3m` is removed after the banding re-check on the training split (§3).
- **Split**: **{split_sizes["train"]:,} train / {split_sizes["test"]:,} test**, stratified by class,
  `random_state = {RANDOM_STATE}`. Stored in the `split` column. The test rows are not used for any decision.

## Features — {len(features)} encoded columns from 23 candidates
| Group | Columns |
|---|---|
| period measures (same period as breakdowns) | {", ".join(f"`{c}`" for c in roles.loc[roles["role"] == "period measure", "feature"])} |
| characteristics | {", ".join(f"`{c}`" for c in roles.loc[roles["role"] == "characteristic", "feature"])} |
| recording flag | {", ".join(f"`{c}`" for c in roles.loc[roles["role"] == "recording flag", "feature"])} |

`Ware_house_ID` is carried as the key only.

## Encoding — fixed rules, nothing learned from rows
- `approved_wh_govt_certificate` → `certificate_grade`: {GRADE}. *Unrated* = 0 is read together with `is_unrated_warehouse`.
- `WH_capacity_size` → `capacity_size`: {CAPACITY}.
- One 0/1 column per non-reference level; reference = most common level in the training split: {REFERENCE}.

## Establishment year
{D4_SPEC}

## Scaling
- PyCaret screening in NB 24 uses normalization so LR/SVM are not disadvantaged by scale.
- The final selected classifier is AdaBoost, fitted in NB 25 on the encoded feature table without a manual scaler.
- `RobustScaler` remains rejected: filled `wh_est_year` has IQR 1, which would leave it with spread 5.47 against 0.50–1.20 for the other scaled columns.

## Shape transformation
{SHAPE_SPEC}

## Characteristics-only feature set
`feature_roles.csv`, column `characteristics_only_set`: {int(roles["characteristics_only_set"].sum())} of {len(features)}
columns (all except the {int((roles["role"] == "period measure").sum())} period measures).
'''
(paths["feature_engine"] / "feature_spec.md").write_text(spec, encoding="utf-8")
print(f"saved  {(paths['feature_engine'] / 'feature_spec.md').relative_to(PROJECT_ROOT)}")

---
## 8. Checks

The saved files are read back. The table must hold every warehouse once, with no gaps, the split and class counts of §2,
no trace of the target's source count, and encodings that reproduce the original categories exactly.

In [ ]:
back = pd.read_csv(paths["processed"] / "classification_base.csv")
roles_back = pd.read_csv(paths["feature_engine"] / "feature_roles.csv")
pre = load_preprocessed().set_index(key).loc[back[key]]

assert back.shape == (25_000, 3 + len(features)) and back[key].is_unique
assert back.notna().all().all()
assert target_source not in back.columns
assert back["split"].value_counts().to_dict() == {"train": 20_000, "test": 5_000}
assert set(back["breakdown_risk"].unique()) == {"Not High Risk", "High Risk"}
assert len(CLASSES) == 2
assert (pd.crosstab(back["breakdown_risk"], back["split"]).reindex(CLASSES)[["train", "test"]].to_numpy()
        == sizes.loc[CLASSES, ["train", "test"]].to_numpy()).all()

assert list(roles_back["feature"]) == features
assert not roles_back.loc[roles_back["characteristics_only_set"], "source_column"].isin(period_measures).any()
assert roles_back.loc[~roles_back["characteristics_only_set"], "source_column"].isin(period_measures).all()

assert (back["certificate_grade"].to_numpy() == pre["approved_wh_govt_certificate"].map(GRADE).to_numpy()).all()
assert ((back["certificate_grade"] == 0).to_numpy() == (pre["is_unrated_warehouse"] == 1).to_numpy()).all()
assert (back["capacity_size"].to_numpy() == pre["WH_capacity_size"].map(CAPACITY).to_numpy()).all()
for c, levels in LEVELS.items():
    cols = [n for n, s in dummy_source.items() if s == c]
    assert back[cols].sum(axis=1).le(1).all()
    is_reference = back[cols].sum(axis=1).eq(0).to_numpy()
    assert (is_reference == (pre[c] == REFERENCE[c]).to_numpy()).all(), c

print("all checks passed")
print(f"classification_base.csv : {back.shape[0]:,} warehouses x {len(features)} features + key, split, target; no gaps")
print(f"feature_roles.csv       : {len(roles_back)} features; characteristics-only set = "
      f"{int(roles_back['characteristics_only_set'].sum())}")
print(f"target classes          : {sorted(back['breakdown_risk'].unique())} ({len(CLASSES)} classes)")

---
## Summary

This notebook built a binary classification table from the preprocessed warehouse data. The target `breakdown_risk`
groups warehouses with 0–3 breakdowns as **Not High Risk** and those with 4–6 breakdowns as **High Risk**, nearly
balanced at a ratio of ~1.09:1. A stratified 80/20 split was applied — 20,000 training and 5,000 test warehouses — and
§3 confirmed from training data alone that the 3|4 boundary is the largest step on all three key features (storage
issues, shipment weight, establishment year). Within-class steps are all smaller, confirming the boundary is informative.

For establishment year, the filled column is kept together with the missing flag. Genuine 2009 warehouses are 60.68%
High Risk, while unknown-year warehouses (filled as 2009) follow the network average. The two groups are
indistinguishable from the year alone; the flag separates them and lifts mutual information from 10.62% to 10.88% of the
class's uncertainty.

Certificate grades and capacity size are encoded as ordered numerics; location type and ownership as 0/1 columns; zone
and regional zone as one-hot columns, each with the most common training level as the reference (Rural, Company Owned,
North, Zone 6). Standard scaling is specified for logistic regression and SVM only — `RobustScaler` is rejected because
the filled year's narrow interquartile range of 1 would leave it with a spread of 5.47 against 0.50–1.20 for the other
scaled columns. No shape transformation is needed; the largest skew among scaled columns is 1.610.

**Output.**

- `data/processed/classification_base.csv` — 25,000 warehouses: key, split, target and **29 encoded features**, unscaled.
- `feature_engine/feature_roles.csv` — each feature's source, role, encoding and scaling. It also marks the
  **characteristics-only set of 24 columns** (all except the five period measures) for the characteristics-only comparison.
- `feature_engine/feature_spec.md` — the same, in words.

**Handed to the feature engineering step:** the duplicate-pair question (storage issues vs shipment weight) and the
negligible-features question, both to be settled on the 20,000 training warehouses only.